# FINAL SQL ANALYSIS

## Introduction
This analysis uses SQL to examine FinTrust Digital Bank's customer and transaction data. It focuses on customer behaviour, transaction activity and value, channel performance, monthly trends, and risk review patterns. The findings will help management, Operations, Digital Banking, and Risk teams understand business performance and identify areas that may need further investigation.

## Objectives
* Review the SQL analyses completed in Weeks 2 and 3.
* Identify overlapping analyses and combine related business questions to reduce repetition.
* Select the most relevant queries based on FinTrust's business objectives and reporting audience.
* Refine the selected queries to provide clear, meaningful business insights.
* Ensure the final queries demonstrate appropriate SQL techniques, including joins, aggregations, conditional logic, CTEs, and window functions where appropriate.

In [1]:
import pandas as pd
import sqlite3

In [3]:
connect = sqlite3.connect(":memory:")

In [8]:
customers = pd.read_csv('customers')
transactions = pd.read_csv('transactions')

customers.to_sql('customers', connect, index=False)
transactions.to_sql('transactions', connect,index=False)

12000

## Customer Distribution Segment

In [9]:
pd.read_sql_query(
    """
    SELECT
        Customer_Segment,
        COUNT(*) AS Customer_Count
    FROM customers
    GROUP BY Customer_Segment
    ORDER BY Customer_Count DESC;
    """,
    connect
)

,Customer_Segment,Customer_Count
0,Everyday,711
1,Premium,295
2,Student,278
3,SME,216


* Everyday customers form the largest segment, accounting for 711 of the 1,500 customers. Premium customers are the second-largest group, followed by Students and SMEs. This shows that the bank's customer base is concentrated in the Everyday segment.

**Business meaning:** Management can use this information to understand the composition of its customer base and plan services and engagement strategies for different customer groups. However, the number of customers in a segment does not automatically mean that the segment generates the most value per customer.

## Transaction Activity and Value by Transaction Type

In [24]:
pd.read_sql_query(
    """
    SELECT
        Transaction_Type,
        COUNT(*) AS Transaction_Count,
        AVG(Amount_NGN) AS Avg_Transaction_Value,
        SUM(Amount_NGN) AS Total_Transaction_Value
    FROM transactions
    GROUP BY Transaction_Type
    ORDER BY Total_Transaction_Value DESC;
    """,
    connect
)

,Transaction_Type,Transaction_Count,Avg_Transaction_Value,Total_Transaction_Value
0,Transfer,3549,67037.672699,2.379167e+08
1,Deposit,1328,98633.328200,1.309851e+08
2,Card Purchase,3033,28309.730887,8.586341e+07
3,Cash Withdrawal,1430,47393.260217,6.777236e+07
4,Bill Payment,1475,19093.998312,2.816365e+07
5,Airtime/Data,1185,8249.933494,9.776171e+06


* Transfers have the highest transaction volume and total value. Card purchases rank second by volume, while deposits generate the second-highest total transaction value despite having fewer transactions than card purchases.
* Deposits also have the highest average transaction value, at approximately NGN 98,633, compared with approximately NGN 67,038 for transfers.

**Business meaning:** Transfers are central to the bank's transaction activity, while deposits account for a comparatively large amount per transaction. Management can use these findings to understand which services drive activity and which contribute the most transaction value.

## Transaction Volume, Value, and Success Rate by Channel

In [11]:
pd.read_sql_query(
    """
    SELECT
        Channel,
        COUNT(*) AS Total_Transactions,
        SUM(Amount_NGN) AS Total_Transaction_Value,
        SUM(
            CASE
                WHEN Transaction_Status = 'Successful' THEN 1
                ELSE 0
            END
        ) AS Successful_Transactions,
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN Transaction_Status = 'Successful' THEN 1
                    ELSE 0
                END
                ) / COUNT(*),
                2
        ) AS Success_Rate
    FROM transactions
    GROUP BY Channel
    ORDER BY Total_Transaction_Value DESC;
    """,
    connect
)

,Channel,Total_Transactions,Total_Transaction_Value,Successful_Transactions,Success_Rate
0,Mobile App,5102,2.401044e+08,4579,89.75
1,POS,2393,1.043467e+08,2174,90.85
2,Web,1869,8.932500e+07,1698,90.85
3,ATM,1747,8.394235e+07,1602,91.70
4,USSD,889,4.275895e+07,803,90.33


* The Mobile App is the most frequently used channel and generates the highest total transaction value. However, it has the lowest success rate at 89.75%. ATM has the highest success rate at 91.70%, although its transaction volume is much lower than the Mobile App's.
* Web and POS have nearly identical success rates.

**Business meaning:** The Mobile App is important to FinTrust's digital banking operations because of its high usage. Its lower success rate makes it a useful priority for further investigation. The results do not establish why transactions fail, so additional analysis would be needed to identify the causes.

## Monthly Transaction Performance

In [13]:
pd.read_sql_query(
    """
    SELECT
        strftime('%m', Transaction_DateTime) AS Month,
        COUNT(Amount_NGN) AS Total_Transactions,
        AVG(Amount_NGN) AS Avg_Total_Transaction,
        SUM(Amount_NGN) AS Total_Transactions_Value,
        ROUND(
            100 * SUM(
                CASE
                    WHEN Transaction_Status = 'Successful' THEN 1
                    ELSE 0
                END) / COUNT(Transaction_ID), 2
            ) AS Success_Rate
    FROM transactions
    GROUP BY Month
    ORDER BY Total_Transactions DESC;
    """,
    connect
)

,Month,Total_Transactions,Avg_Total_Transaction,Total_Transactions_Value,Success_Rate
0,03,4133,47471.817849,1.962010e+08,90.0
1,01,4133,45605.960486,1.884894e+08,90.0
2,02,3734,47077.369306,1.757869e+08,90.0


* Transaction volume declined in February from 4,133 to 3,734, then recovered to 4,133 in March. Total transaction value followed the same general pattern, falling in February before reaching its highest level in March. Average transaction value increased across the three months, while the success rate remained stable at approximately 90%.

**Business meaning:** FinTrust experienced a temporary decline in transaction activity in February, followed by a recovery in March. The increasing average transaction value suggests that the amount processed per transaction rose over the period. Management can monitor future months to determine whether these patterns continue.

## Customer Transaction Behaviour by Segment

In [14]:
pd.read_sql_query(
    """
    SELECT
        c.Customer_Segment,
        COUNT(t.Transaction_ID) AS Total_Transactions,
        COUNT(DISTINCT t.Customer_ID) AS Active_Customers,
        SUM(t.Amount_NGN) AS Total_Transaction_Value,
        AVG(t.Amount_NGN) AS AVG_Transaction_Value,
        SUM(t.Amount_NGN) / COUNT(DISTINCT t.Customer_ID) AS Value_Per_Active_Customer
    FROM transactions t
    JOIN customers c
        ON t.Customer_ID = c.Customer_ID
    GROUP BY c.Customer_Segment
    ORDER BY Total_Transactions DESC;
    """,
    connect
)

,Customer_Segment,Total_Transactions,Active_Customers,Total_Transaction_Value,AVG_Transaction_Value,Value_Per_Active_Customer
0,Everyday,5644,711,2.614589e+08,46325.110027,367734.066090
1,Premium,2361,295,1.077512e+08,45637.948251,365258.290915
2,Student,2289,278,1.074759e+08,46953.196300,386603.835719
3,SME,1706,216,8.379137e+07,49115.692679,387923.017176


* Everyday customers account for the most transactions and the highest total transaction value, largely because they are the largest customer segment. However, SMEs have the highest average transaction value and the highest total transaction value per active customer, at approximately NGN 387,923. Students follow closely at approximately NGN 386,604 per active customer.
* Premium customers have the lowest value per active customer among the four segments.

**Business meaning:** Looking only at total transaction value would make the Everyday segment appear to be the strongest contributor. Comparing value per active customer provides a different perspective, showing that SMEs and Students generate more transaction value per active customer in this dataset. Management could use this insight to investigate the needs and service opportunities of these segments.

## Customer-Level Transaction Frequency and Value

In [15]:
pd.read_sql_query(
    """
    SELECT
        c.Customer_ID,
        c.Customer_Segment,
        SUM(t.AMOUNT_NGN) AS Total_Transaction_Value,
        COUNT(t.Transaction_ID) AS Total_Transactions,
        AVG(t.AMOUNT_NGN) AS Avg_Transaction_Value
    FROM customers c
    JOIN transactions t
        ON c.Customer_ID = t.Customer_ID
    GROUP BY
        c.Customer_ID,
        c.Customer_Segment
    ORDER BY Total_Transactions DESC;
    """,
    connect
)

,Customer_ID,Customer_Segment,Total_Transaction_Value,Total_Transactions,Avg_Transaction_Value
0,FT-C01296,Everyday,783519.64,20,39175.982000
1,FT-C00965,Student,376271.52,19,19803.764211
2,FT-C00209,SME,1063845.37,18,59102.520556
3,FT-C00351,SME,649777.49,18,36098.749444
4,FT-C00357,Everyday,1713516.28,18,95195.348889
...,...,...,...,...,...
1495,FT-C00923,Everyday,56024.67,2,28012.335000
1496,FT-C01279,Premium,287197.19,2,143598.595000
1497,FT-C00520,Everyday,208519.26,1,208519.260000
1498,FT-C00983,Everyday,24725.37,1,24725.370000


* Customer activity is not evenly distributed. Some customers conduct very few transactions, while others account for much more frequent activity.
Transaction value also varies between customers.

**Business meaning:** This analysis gives management a more detailed view than segment-level averages alone. It can support customer grouping by activity and transaction value, help identify highly active customers, and inform engagement strategies.

## Risk Review by Transaction Type

In [20]:
pd.read_sql_query(
    """
    SELECT
        Transaction_Type,
        COUNT(*) AS Total_Transactions,
        SUM(
            CASE
                WHEN Risk_Review_Flag = 'Yes' THEN 1
                ELSE 0
            END
        ) AS Risk_Review_Count,
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN Risk_Review_Flag = 'Yes' THEN 1
                    ELSE 0
                END) / COUNT(*),
                2
            ) AS Risk_Review_Rate
    FROM transactions
    GROUP BY Transaction_Type
    ORDER BY Risk_Review_Rate DESC;
    """,
    connect
)

,Transaction_Type,Total_Transactions,Risk_Review_Count,Risk_Review_Rate
0,Transfer,3549,1011,28.49
1,Cash Withdrawal,1430,362,25.31
2,Deposit,1328,215,16.19
3,Airtime/Data,1185,180,15.19
4,Card Purchase,3033,395,13.02
5,Bill Payment,1475,189,12.81


* Transfers have the highest risk review rate, followed by cash withdrawals. Bill payments and card purchases have the lowest rates among the transaction types listed.

**Business meaning:** The Risk team could prioritise further investigation of transfers and cash withdrawals to understand why they are reviewed more frequently. These figures represent risk review flags, not confirmed fraud, so they should not be interpreted as proof that these transaction types are unsafe.

## International Vs. Domestic Transaction Value and Risk

In [21]:
pd.read_sql_query(
    """
    SELECT
        International_Transaction,
        SUM(Amount_NGN) AS Total_Transaction_Value,
        COUNT(Transaction_ID) AS Total_Transactions,
        SUM(
            CASE
                WHEN Risk_Review_Flag = 'Yes' THEN 1
                ELSE 0
            END) AS Risk_Reviews,
        ROUND(
            100.0 *  SUM(
                CASE
                    WHEN Risk_Review_Flag = 'Yes' THEN 1
                    ELSE 0
                END) / COUNT(Transaction_ID), 2
            ) AS Risk_Review_Rate
    FROM transactions
    GROUP BY International_Transaction;
    """,
    connect
)

,International_Transaction,Total_Transaction_Value,Total_Transactions,Risk_Reviews,Risk_Review_Rate
0,No,5.404816e+08,11520,2175,18.88
1,Yes,1.999578e+07,480,177,36.88


* International transactions account for only 480 of the 12,000 transactions, yet their risk review rate is 36.88%, compared with 18.88% for domestic transactions.
* Domestic transactions account for more risk reviews in absolute numbers because their transaction volume is much larger.

**Business meaning:** International transactions are a useful priority for risk investigation because a larger proportion of them is flagged for review. The Risk team should investigate the factors associated with this difference before changing monitoring rules or treating international transactions as inherently risky.

## Risk Review by Customer Segment

In [22]:
pd.read_sql_query(
    """
    SELECT
        c.Customer_Segment,
        COUNT(t.TRANSACTION_ID) AS Total_Transactions,
        SUM(
            CASE
                WHEN Risk_Review_Flag = 'Yes' THEN 1
                ELSE 0
            END) AS Risk_Reviews,
        ROUND(
            100.0 *  SUM(
                CASE
                    WHEN Risk_Review_Flag = 'Yes' THEN 1
                    ELSE 0
                END) / COUNT(t.Transaction_ID), 2
            ) AS Risk_Review_Rate
    FROM transactions t
    JOIN customers c
        ON t.Customer_ID = c.Customer_ID
    GROUP BY c.Customer_Segment
    ORDER BY Risk_Review_Rate DESC;
    """,
    connect
)

,Customer_Segment,Total_Transactions,Risk_Reviews,Risk_Review_Rate
0,Premium,2361,480,20.33
1,Student,2289,452,19.75
2,SME,1706,331,19.40
3,Everyday,5644,1089,19.29


* Premium customers have the highest risk review rate at 20.33%, while Everyday customers have the lowest at 19.29%. However, the difference between the highest and lowest rates is only about one percentage point.
* Everyday customers account for the highest number of risk reviews because they also have the highest transaction volume.

**Business meaning:** Risk review rates are relatively similar across the four customer segments. This suggests that customer segment alone may not be sufficient to explain risk review patterns. Further investigation should consider transaction type, channel, international status, and transaction amount alongside customer segment.

## High-Value Transaction Analysis

In [23]:
pd.read_sql_query(
    """
    SELECT
        Transaction_Type,
        COUNT(Transaction_ID) AS Total_Transactions,
        SUM(Amount_NGN) AS Transactions_Value,
        AVG(Amount_NGN) AS Avg_Transaction_Value
    FROM transactions
    WHERE Amount_NGN > 117922.86
    GROUP BY Transaction_Type
    ORDER BY Transactions_Value DESC;
    """,
    connect
)

,Transaction_Type,Total_Transactions,Transactions_Value,Avg_Transaction_Value
0,Transfer,718,1.761903e+08,245390.346964
1,Deposit,344,1.077435e+08,313207.860552
2,Cash Withdrawal,209,3.631795e+07,173770.098517
3,Card Purchase,203,2.988171e+07,147200.542315


* Transfers account for the highest number of transactions above the threshold and the highest total value among the types listed. Deposits have the highest average value, at approximately NGN 313,208 per transaction.
* This shows that high-value transaction activity is concentrated differently depending on whether we look at the number of transactions, total value, or average value.

**Business meaning:** Transfers deserve attention because of their high count and total value, while deposits stand out for their higher average transaction amount. The Risk team can use the threshold to identify transactions for further investigation, but exceeding the IQR threshold does not automatically mean a transaction is fraudulent or incorrect.